# Customer Segmentation using RFM Analysis & K-Means Clustering

**Objective:** Segment ~540K eCommerce transactions into actionable customer groups using Recency, Frequency, and Monetary (RFM) metrics combined with K-Means clustering — enabling targeted marketing strategies for each segment.

**Dataset:** [UCI Online Retail Dataset](https://archive.ics.uci.edu/ml/datasets/online+retail) — 541,909 transactions from a UK-based retailer (Dec 2010 – Dec 2011).

**Key Findings:**
- 4 distinct customer segments identified (VIP, Loyal, Potential, At-Risk)
- A small group of high-value customers drives the majority of revenue
- Frequency and Monetary are strongly correlated (0.57), while Recency is negatively correlated with both
- Peak ordering activity occurs mid-week and during late morning hours

---
## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from mpl_toolkits.mplot3d import Axes3D

sns.set(style='whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

df = pd.read_csv('data.csv', encoding='ISO-8859-1')
print(f'Dataset shape: {df.shape}')
df.head()

---
## 2. Data Cleaning & Preprocessing

Key steps: convert date types, handle missing CustomerIDs (required for RFM), and engineer the `TotalPrice` feature.

In [ ]:
# Date conversion
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Check missing values before cleaning
print('Missing values before cleaning:')
print(df.isnull().sum())
print(f'\nTotal rows: {len(df)}')

# Drop rows without CustomerID (can't compute RFM without it)
df = df.dropna(subset=['CustomerID'])

# Remove duplicates
df = df.drop_duplicates()

# Feature engineering
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print(f'\nCleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Unique customers: {df["CustomerID"].nunique()}')
print(f'Date range: {df["InvoiceDate"].min().date()} to {df["InvoiceDate"].max().date()}')

---
## 3. RFM Metric Calculation

For each customer:
- **Recency** — days since last purchase (lower = better)
- **Frequency** — count of unique invoices (higher = better)
- **Monetary** — total spend (higher = better)

In [ ]:
reference_date = df['InvoiceDate'].max()

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

rfm.set_index('CustomerID', inplace=True)
print(f'RFM table: {rfm.shape[0]} customers')
rfm.describe().round(2)

### RFM Distributions

All three metrics are right-skewed — most customers have low frequency and spend, while a small group of power users drives disproportionate value.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col, color in zip(axes, ['Recency', 'Frequency', 'Monetary'], ['#3498db', '#2ecc71', '#e74c3c']):
    sns.histplot(rfm[col], bins=30, kde=True, ax=ax, color=color)
    ax.set_title(f'{col} Distribution')

plt.tight_layout()
plt.show()

---
## 4. RFM Scoring (Quartile-Based)

Each metric is scored 1–4 using quartiles. For Recency, a lower value is better so the scoring is inverted.

In [ ]:
rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), 4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4])

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

rfm.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['R_Score', 'F_Score', 'M_Score']):
    sns.countplot(x=col, data=rfm, ax=ax, palette='Blues_d')
    ax.set_title(f'{col} Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Average Monetary by Recency x Frequency scores
rfm_heatmap = rfm.pivot_table(index='R_Score', columns='F_Score', values='Monetary', aggfunc='mean')

plt.figure(figsize=(8, 6))
sns.heatmap(rfm_heatmap, annot=True, fmt='.0f', cmap='YlGnBu')
plt.title('Average Monetary by Recency × Frequency Score')
plt.show()

---
## 5. K-Means Clustering

Using the Elbow Method to find the optimal number of clusters, then applying K-Means to group customers into natural segments.

In [ ]:
# Prepare and scale features
rfm_numeric = rfm[['R_Score', 'F_Score', 'M_Score']].astype(float)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_numeric)

# Elbow method
inertia = []
K = range(2, 11)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K, inertia, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Optimal k')
plt.show()

In [ ]:
# Apply K-Means with k=4 (based on elbow plot)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Cluster summary
cluster_summary = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(1)
cluster_summary['Count'] = rfm.groupby('Cluster').size()
cluster_summary

---
## 6. Cluster Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Cluster', palette='Set2', ax=axes[0], alpha=0.7)
axes[0].set_title('Recency vs Monetary')

sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Cluster', palette='Set2', ax=axes[1], alpha=0.7)
axes[1].set_title('Frequency vs Monetary')

plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    rfm['Recency'], rfm['Frequency'], rfm['Monetary'],
    c=rfm['Cluster'], cmap='Set2', s=40, alpha=0.7
)

ax.set_xlabel('Recency')
ax.set_ylabel('Frequency')
ax.set_zlabel('Monetary')
plt.title('3D RFM Clusters')
plt.legend(*scatter.legend_elements(), title='Cluster')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['Recency', 'Frequency', 'Monetary']):
    sns.boxplot(data=rfm, x='Cluster', y=col, ax=ax, palette='Set2')
    ax.set_title(f'{col} by Cluster')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(rfm[['Recency', 'Frequency', 'Monetary']].corr(), annot=True, cmap='Blues', fmt='.2f')
plt.title('RFM Correlation Matrix')
plt.show()

In [ ]:
# Radar chart comparing cluster profiles
cluster_means = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()
labels = ['Recency', 'Frequency', 'Monetary']
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw={'polar': True})
colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']

for idx, (cluster, ax) in enumerate(zip(cluster_means.index, axes.flat)):
    values = cluster_means.loc[cluster].tolist()
    values += values[:1]
    angle_vals = angles + angles[:1]

    ax.plot(angle_vals, values, linewidth=2, color=colors[idx])
    ax.fill(angle_vals, values, alpha=0.25, color=colors[idx])
    ax.set_xticks(angles)
    ax.set_xticklabels(labels)
    ax.set_title(f'Cluster {cluster}', size=13, fontweight='bold')

plt.tight_layout()
plt.show()

### Pairplot — Full RFM Scatter Matrix

In [ ]:
sns.pairplot(rfm[['Recency', 'Frequency', 'Monetary', 'Cluster']], hue='Cluster', palette='Set2')
plt.suptitle('RFM Pairplot by Cluster', y=1.02)
plt.show()

---
## 7. Segment Profiling & Marketing Recommendations

Labeling each cluster based on its RFM characteristics and suggesting targeted strategies.

In [ ]:
rfm['R_Score'] = pd.to_numeric(rfm['R_Score'])
rfm['F_Score'] = pd.to_numeric(rfm['F_Score'])
rfm['M_Score'] = pd.to_numeric(rfm['M_Score'])

segment_profile = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score']].mean().round(2)
segment_profile['Customer_Count'] = rfm.groupby('Cluster').size()

print('SEGMENT PROFILES')
display(segment_profile)

In [ ]:
# Segment labels and strategies (adjust cluster numbers based on your output)
segment_strategies = {
    0: {
        'label': 'VIP / Champions',
        'profile': 'Low recency, high frequency, highest spend',
        'strategy': 'Exclusive perks, early access to new products, personalized recommendations, VIP loyalty tier'
    },
    1: {
        'label': 'Loyal Customers',
        'profile': 'Moderate recency, moderate frequency, good spend',
        'strategy': 'Loyalty rewards program, upsell/cross-sell, seasonal discounts, referral incentives'
    },
    2: {
        'label': 'At-Risk / Churning',
        'profile': 'High recency, low frequency, low spend',
        'strategy': 'Win-back email campaigns, "We miss you" offers, feedback surveys, time-limited discounts'
    },
    3: {
        'label': 'New / Potential',
        'profile': 'Low-moderate recency, low frequency, low spend',
        'strategy': 'Welcome sequences, bundle offers, retargeting ads, onboarding content'
    }
}

for cluster, info in segment_strategies.items():
    print(f"\nCluster {cluster} — {info['label']}")
    print(f"  Profile:  {info['profile']}")
    print(f"  Strategy: {info['strategy']}")

---
## 8. Extended Analysis

Additional exploration of ordering patterns, geography, product performance, returns, and customer behavior.

### 8.1 Time-Based Patterns

In [ ]:
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['HourOfDay'] = df['InvoiceDate'].dt.hour
df['Month'] = df['InvoiceDate'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Orders by day of week
day_order = df.groupby('DayOfWeek')['InvoiceNo'].nunique().sort_values(ascending=False)
day_order.plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Orders by Day of Week')
axes[0].set_ylabel('Number of Orders')

# Orders by hour
hour_order = df.groupby('HourOfDay')['InvoiceNo'].nunique()
hour_order.plot(kind='bar', ax=axes[1], color='orange')
axes[1].set_title('Orders by Hour of Day')
axes[1].set_ylabel('Number of Orders')

# Monthly trend
monthly = df.groupby('Month')['InvoiceNo'].nunique()
monthly.plot(kind='line', marker='o', ax=axes[2], color='green')
axes[2].set_title('Monthly Orders Trend')
axes[2].set_ylabel('Number of Orders')

plt.tight_layout()
plt.show()

### 8.2 Geographic Distribution

In [ ]:
top5_countries = df.groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=False).head(5)

plt.figure(figsize=(8, 5))
top5_countries.plot(kind='bar', color='skyblue')
plt.title('Top 5 Countries by Order Volume')
plt.ylabel('Number of Orders')
plt.xticks(rotation=45)
plt.show()

print(top5_countries)

### 8.3 Product Analysis

In [ ]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Top 10 products by quantity sold
top10 = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
print('Top 10 Products by Quantity Sold:')
print(top10)

# Top revenue-generating product
top_revenue = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(5)

plt.figure(figsize=(10, 5))
top_revenue.plot(kind='bar', color='green')
plt.title('Top 5 Products by Revenue')
plt.ylabel('Revenue ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 8.4 Returns Analysis

In [ ]:
return_orders = df[df['Quantity'] < 0]['InvoiceNo'].nunique()
total_orders = df['InvoiceNo'].nunique()
return_pct = (return_orders / total_orders) * 100

print(f'Return rate: {return_pct:.2f}% of all orders')

# Products with highest return rates
df['IsReturn'] = df['Quantity'] < 0
return_rate = df.groupby('Description')['IsReturn'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
return_rate.plot(kind='bar', color='#e74c3c')
plt.title('Top 10 Products by Return Rate')
plt.ylabel('Return Rate')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 8.5 Customer Behavior

In [ ]:
# Distribution of orders per customer
orders_per_customer = df.groupby('CustomerID')['InvoiceNo'].nunique()

print(f'Average orders per customer: {orders_per_customer.mean():.1f}')
print(f'Median orders per customer: {orders_per_customer.median():.0f}')

# Repeat purchase rate
repeat_pct = (orders_per_customer.gt(1).sum() / len(orders_per_customer)) * 100
print(f'Repeat purchase rate: {repeat_pct:.1f}%')

# Customer active duration
activity = df.groupby('CustomerID')['InvoiceDate'].agg(['min', 'max'])
activity['ActiveDays'] = (activity['max'] - activity['min']).dt.days

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(orders_per_customer, bins=range(1, orders_per_customer.max() + 2), edgecolor='black')
axes[0].set_xlabel('Number of Orders')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Distribution of Orders per Customer')

axes[1].hist(activity['ActiveDays'], bins=30, edgecolor='black', color='#2ecc71')
axes[1].set_xlabel('Active Days')
axes[1].set_ylabel('Number of Customers')
axes[1].set_title('Customer Active Duration')

plt.tight_layout()
plt.show()

---
## Key Takeaways

1. **4 actionable customer segments** were identified — each requiring a different marketing approach, from VIP retention programs to win-back campaigns for at-risk customers.

2. **Revenue concentration is extreme** — a small percentage of high-value customers drives the majority of total revenue, reinforcing the importance of targeted retention.

3. **Strong Frequency–Monetary correlation (0.57)** confirms that customers who buy more often also spend more — loyalty programs that incentivize repeat purchases should directly increase revenue.

4. **Temporal and geographic patterns** provide tactical optimization opportunities: mid-week promotions, late-morning engagement windows, and prioritizing the UK market with selective international expansion.

---

**Potential extensions:** Customer Lifetime Value (CLV) modeling, predictive churn analysis, DBSCAN/hierarchical clustering comparison, real-time segmentation dashboard.